# Chunk size and retrieval experiments

This notebook tests how chunk size and the optional local reranker affect retrieval, using the 15 build-set questions in `data/eval/questions.json`.

For each configuration it indexes the build documents into a **temporary** Chroma folder (the real `vector_db/` is not touched), then runs retrieval only. No answers are generated.

Measures, on the 10 answerable questions unless noted:

- **hit@1 / hit@3**: share of questions whose expected passage is ranked first / in the top 3.
- **MRR**: mean reciprocal rank of the expected passage.
- **role leaks**: chunks returned to a role that may not see them (all 15 questions).
- **score margin**: lowest top-1 relevance of an answerable question minus the highest top-1 relevance of an unanswerable one. A positive margin would mean a relevance threshold alone could separate them.

Running it sends the synthetic document text and questions to the embedding provider configured in `.env`.

In [1]:
import shutil
import sys
import tempfile
from functools import lru_cache
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from app.evaluation.evaluator import evaluate_retrieval, load_cases
from app.evaluation.metrics import hit_rate, mean_reciprocal_rank
from app.ingestion.chunking import chunk_document, load_document_rules
from app.ingestion.embedder import get_embedder
from app.ingestion.loader import load_document
from app.retrieval.reranker import LexicalReranker
from app.retrieval.retriever import Retriever
from app.retrieval.vector_store import VectorStore
from run_ingestion import build_index, default_roots, discover_files

questions, _ = load_cases()
rules = load_document_rules()
documents = [load_document(path) for path in discover_files(default_roots()) if path.name in rules]
documents = [document for document in documents if not document.needs_ocr]
embedder = get_embedder()
embedder.embed_query = lru_cache(maxsize=None)(embedder.embed_query)
WORK_DIR = Path(tempfile.mkdtemp(prefix="privatedesk-experiments-"))
print(f"{len(documents)} documents, {len(questions)} questions, embeddings: {embedder.provider}/{embedder.model}")

12 documents, 15 questions, embeddings: gemini/gemini-embedding-2


In [2]:
def build_store(size, overlap):
    chunks = [
        chunk
        for document in documents
        for chunk in chunk_document(document, rules[document.document], size=size, overlap=overlap)
    ]
    store = VectorStore(path=WORK_DIR / f"chunks_{size}_{overlap}")
    build_index(chunks, embedder=embedder, store=store)
    return store, len(chunks)


def evaluate(store, reranker):
    retriever = Retriever(embedder=embedder, store=store, top_k=5, reranker=reranker)
    results = evaluate_retrieval(retriever, questions, k=5)
    ranks = [result.rank for result in results if result.answerable]
    top_score = {case["id"]: retriever.retrieve(case["question"], case["role"], 1)[0].score for case in questions}
    lowest_answerable = min(top_score[case["id"]] for case in questions if case["answerable"])
    highest_unanswerable = max(top_score[case["id"]] for case in questions if not case["answerable"])
    summary = {
        "hit@1": hit_rate(ranks, 1),
        "hit@3": hit_rate(ranks, 3),
        "MRR": round(mean_reciprocal_rank(ranks), 3),
        "role leaks": sum(len(result.leaks) for result in results),
        "score margin": round(lowest_answerable - highest_unanswerable, 3),
    }
    return summary, {result.case_id: result.rank for result in results if result.answerable}

In [3]:
CONFIGS = [(500, 80), (900, 150), (1400, 200)]
RERANKERS = {"none": None, "lexical": LexicalReranker()}

rows, ranks = [], {}
for size, overlap in CONFIGS:
    store, count = build_store(size, overlap)
    for name, reranker in RERANKERS.items():
        summary, question_ranks = evaluate(store, reranker)
        rows.append({"chunk size": size, "overlap": overlap, "chunks": count, "reranker": name, **summary})
        ranks[f"{size}/{name}"] = question_ranks

results = pd.DataFrame(rows)
results

,chunk size,overlap,chunks,reranker,hit@1,hit@3,MRR,role leaks,score margin
0,500,80,61,none,1.0,1.0,1.00,0,-0.014
1,500,80,61,lexical,0.9,1.0,0.95,0,-0.025
2,900,150,37,none,1.0,1.0,1.00,0,-0.005
3,900,150,37,lexical,0.9,1.0,0.95,0,-0.008
4,1400,200,22,none,0.9,1.0,0.95,0,-0.006
5,1400,200,22,lexical,0.7,1.0,0.85,0,-0.012


Rank of the expected passage for each answerable question (1 = ranked first, missing = not in the top 5):

In [4]:
pd.DataFrame(ranks).fillna("miss")

,500/none,500/lexical,900/none,900/lexical,1400/none,1400/lexical
q01,1,1,1,1,1,2
q02,1,1,1,1,1,1
q03,1,1,1,1,1,1
q04,1,1,1,1,1,1
q05,1,1,1,1,1,1
q06,1,1,1,1,1,1
q07,1,1,1,1,1,1
q08,1,1,1,1,1,2
q09,1,1,1,1,2,1
q10,1,2,1,2,1,2


## Findings

Run on 2026-10-25 with `gemini-embedding-2` (768 dimensions) over the 12 build documents that have text.

1. **Chunk size.** 500 and 900 characters both rank the expected passage first for all 10 answerable questions. At 1400 characters, q09 (the ERP go-live date) drops to rank 2, because larger chunks mix several topics. The app keeps **900 / 150**: the same accuracy as 500 with 37 chunks instead of 61.
2. **Lexical reranker.** It lowers hit@1 at every chunk size (1.0 to 0.9 at 500 and 900, 0.9 to 0.7 at 1400), while hit@3 stays at 1.0. On q10 it promotes passages that repeat the question's words ("Dubai Investments Park") over the passage that contains the move date. Under the plan's rule of keeping a change only if the score rises, the reranker stays **off** (`RERANKER=none`).
3. **Score margin is negative in every configuration.** Some unanswerable questions, such as the Q3 revenue near-miss, score as high as answerable ones. A relevance threshold alone cannot decide "not found", so refusal is left to the grounded prompt and the citation check, and `MIN_RELEVANCE=0.55` only screens out clearly off-topic questions.
4. **No configuration leaked a chunk across roles.**

Caveat: 10 answerable questions over 12 synthetic documents is a small sample. A change of 0.1 in hit@1 is a single question.

In [5]:
shutil.rmtree(WORK_DIR, ignore_errors=True)